# Image denoising

Apply ANTsTorch's non-local means denoiser to the real `r16` image used in the
[ANTsX tutorial](https://gist.github.com/ntustison/12a656a5fc2f6f9c4494c88dc09c5621). ANTsPy downloads this sample on first use. No simulated image or
artificial noise is added. Inspect the filtered image and removed residual;
there is no noise-free reference for measuring reconstruction error.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

In [ ]:
image = ants.image_read(ants.get_ants_data("r16")).clone("float")
mask = ants.get_mask(image)
denoised = antstorch.denoise_image(
    image, mask=mask, noise_model="Rician", p=1, r=2, shrink_factor=1, device=device,
)
original = image.numpy()
output = denoised.numpy()
assert output.shape == original.shape and np.isfinite(output).all()
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(denoised, attribute), getattr(image, attribute))
residual = original - output
inside = mask.numpy() > 0
print("Mean absolute intensity change inside mask:", float(np.abs(residual[inside]).mean()))

## Inspect filtered anatomy and the residual
A Rician noise model is selected for this magnitude-MRI example; assess whether
it fits your acquisition. Patch radius `p` and search radius `r` affect smoothing,
detail preservation and run time. The residual can contain anatomical detail as
well as noise and must not be treated as a ground-truth noise map.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
for ax, array, title in zip(axes, [original, output, residual],
                            ["Tutorial r16", "Denoised", "Removed residual"]):
    view = ax.imshow(array.T, origin="lower", cmap="gray",
                     vmin=None if title == "Removed residual" else original.min(),
                     vmax=None if title == "Removed residual" else original.max())
    ax.set_title(title); ax.axis("off")
    fig.colorbar(view, ax=ax, shrink=0.7)
plt.show()

Inspect edges and small structures after filtering. Reduced residual variance alone does not establish preserved anatomy.